In [1]:
import pandas as pd
import json
import requests
import time

# 1. โหลดและเตรียมข้อมูล
df = pd.read_csv('telemetry_data.csv')
df_target = df[df['device_id'] == 'mix-pump-101']

api_url = "http://localhost:8002/ingest/analyze"
window_size = 24*4  # ขนาดของ Sliding Window

print(f"จำนวนข้อมูลทั้งหมด {len(df_target)} แถว")
print(f"จะทำการเลื่อน (Sliding) ทีละ 1 แถว โดยใช้ Window Size = {window_size}")
print("="*60)

# 2. วนลูปแบบ Sliding Window
# ใช้ range จนถึง (จำนวนแถวทั้งหมด - ขนาด window + 1) เพื่อป้องกันการดึงข้อมูลเกินขอบเขต
for i in range(len(df_target) - window_size + 1):
    
    # สไลซ์ข้อมูลออกมาตามขนาด window_size (เลื่อนทีละ 1 index)
    window_data = df_target.iloc[i : i + window_size]
    
    # แปลงเป็น JSON List of Dicts
    data_json = window_data.to_json(orient='records')
    parsed_window = json.loads(data_json)
    
    # ดึงเวลาของข้อมูลตัวล่าสุดใน Window มาใช้สำหรับแสดงผลอ้างอิง
    latest_timestamp = parsed_window[-1].get('timestamp', 'Unknown')
    
    print(f"[Window {i+1}] Index {i} ถึง {i + window_size - 1} ")
    
    # 3. ส่งข้อมูล 1 Sequence (10 รายการ) ไปยัง API
    try:
        # ส่ง payload เป็น Array ของ 10 records
        response = requests.post(api_url, json=parsed_window)
        response.raise_for_status()
        
        result = response.json()
        
        # ดึงผลการวิเคราะห์มาแสดง
        risk = result.get('analysis', {}).get('risk_level', 'N/A')
        anomaly_score = result.get('analysis', {}).get('anomaly_score', 'N/A')
        
        print(f"  -> ผลลัพธ์: Risk = {risk} | Anomaly Score = {anomaly_score}")
        
    except requests.exceptions.RequestException as e:
         print(f"  -> เกิดข้อผิดพลาดในการเรียก API: {e}")
         
    # พักระยะเวลาสั้นๆ ก่อนส่ง Window ถัดไป (ปรับเพิ่ม/ลดได้ตามต้องการ)
    time.sleep(0.5)

print("\n" + "="*60)
print("วิเคราะห์ข้อมูลแบบ Sliding Window เสร็จสิ้น!")

จำนวนข้อมูลทั้งหมด 6732 แถว
จะทำการเลื่อน (Sliding) ทีละ 1 แถว โดยใช้ Window Size = 96
[Window 1] Index 0 ถึง 95 
  -> ผลลัพธ์: Risk = low | Anomaly Score = 0.125
[Window 2] Index 1 ถึง 96 
  -> ผลลัพธ์: Risk = low | Anomaly Score = 0.125
[Window 3] Index 2 ถึง 97 


KeyboardInterrupt: 